# BanglaLLM — Local Data Pipeline
Runs entirely on your local machine. No Google Drive or Colab required.

**Expected folder layout:**
```
bengali_datasets/
├── titulm_local/        ← your .arrow files
└── local_mc4_bn_data/   ← your .arrow files
```

In [ ]:
# # ─────────────────────────────────────────────────────────────
# # CELL 1 — Install dependencies (run once)
# # ─────────────────────────────────────────────────────────────
# !pip install -q datasets sentencepiece transformers pyarrow huggingface_hub

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Set all folder paths
# ─────────────────────────────────────────────────────────────

import os

# ── INPUT: Your local dataset folder ─────────────────────────
DATASET_FOLDER   = "./bengali_datasets"     # contains titulm_local/ and local_mc4_bn_data/

# ── OUTPUT: All generated files go here ──────────────────────
CLEANED_OUTPUT   = "./BanglaLLM/cleaned"
TOKENIZER_OUTPUT = "./BanglaLLM/tokenizer"
FINAL_OUTPUT     = "./BanglaLLM/final_dataset"

os.makedirs(CLEANED_OUTPUT,   exist_ok=True)
os.makedirs(TOKENIZER_OUTPUT, exist_ok=True)
os.makedirs(FINAL_OUTPUT,     exist_ok=True)

print("All output folders created.")

All output folders created.


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Find all .arrow files
# ─────────────────────────────────────────────────────────────

import glob

arrow_files = glob.glob(f"{DATASET_FOLDER}/**/*.arrow", recursive=True)

print(f"Found {len(arrow_files)} .arrow file(s):\n")
for f in arrow_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

Found 4 .arrow file(s):

  ./bengali_datasets/local_mc4_bn_data/c4-train-00000-of-00093.arrow  (501.2 MB)
  ./bengali_datasets/local_mc4_bn_data/c4-train-00001-of-00093.arrow  (505.4 MB)
  ./bengali_datasets/titulm_local/data-00000-of-00295.arrow  (500.4 MB)
  ./bengali_datasets/titulm_local/data-00001-of-00295.arrow  (556.1 MB)


In [7]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Load .arrow files into a HuggingFace Dataset
#
# Method A: HuggingFace save_to_disk() format
#           (has dataset_info.json + state.json)
# Method B: Raw .arrow files with no metadata JSON
# ─────────────────────────────────────────────────────────────

from datasets import Dataset, load_from_disk, concatenate_datasets
import pyarrow as pa

def load_arrow_data(dataset_folder):
    """
    Loads .arrow files from dataset_folder.
    Handles both HuggingFace save_to_disk format and raw arrow files.
    Returns a HuggingFace Dataset with a 'text' column.
    """

    # ── Method A: HuggingFace save_to_disk() format ──────────
    hf_dataset_folders = []
    for root, dirs, files in os.walk(dataset_folder):
        if "dataset_info.json" in files:
            hf_dataset_folders.append(root)

    if hf_dataset_folders:
        print(f"Found {len(hf_dataset_folders)} HuggingFace dataset folder(s):")
        loaded = []
        for folder in hf_dataset_folders:
            print(f"  Loading: {folder}")
            try:
                ds = load_from_disk(folder)
                print(f"    -> {len(ds):,} rows | columns: {ds.column_names}")
                loaded.append(ds)
            except Exception as e:
                print(f"    -> Failed: {e}")

        if loaded:
            combined = concatenate_datasets(loaded) if len(loaded) > 1 else loaded[0]
            if "text" not in combined.column_names:
                for col in ["content", "body", "sentence", "passage"]:
                    if col in combined.column_names:
                        combined = combined.rename_column(col, "text")
                        print(f"Renamed column '{col}' -> 'text'")
                        break
            return combined

    # ── Method B: Raw .arrow files (no metadata JSON) ────────
    arrow_files = glob.glob(f"{dataset_folder}/**/*.arrow", recursive=True)
    if not arrow_files:
        raise FileNotFoundError(
            f"No .arrow files found in {dataset_folder}\n"
            f"Please check that DATASET_FOLDER points to the right path."
        )

    print(f"Loading {len(arrow_files)} raw .arrow file(s) with PyArrow...")
    all_tables = []
    for filepath in arrow_files:
        print(f"  Reading: {os.path.basename(filepath)}")
        try:
            with pa.ipc.open_stream(filepath) as reader:
                table = reader.read_all()
        except pa.ArrowInvalid:
            try:
                with pa.ipc.open_file(filepath) as reader:
                    table = reader.read_all()
            except pa.ArrowInvalid:
                print(f"    -> Could not read {filepath}, skipping.")
                continue

        print(f"    -> {table.num_rows:,} rows | columns: {table.schema.names}")

        # ── Normalise: keep only the 'text' column ────────────
        # Each source dataset may have different extra columns
        # (e.g. timestamp+url vs document_id). Strip everything
        # except 'text' so all tables share the same schema.
        text_col = None
        for candidate in ["text", "content", "body", "sentence", "passage"]:
            if candidate in table.schema.names:
                text_col = candidate
                break

        if text_col is None:
            # Fall back to first string column
            for i, field in enumerate(table.schema):
                if pa.types.is_string(field.type) or pa.types.is_large_string(field.type):
                    text_col = field.name
                    break

        if text_col is None:
            print(f"    -> No text column found, skipping.")
            continue

        if text_col != "text":
            print(f"    -> Renaming column '{text_col}' -> 'text'")

        # Select only the text column and cast to plain string
        text_array = table.column(text_col).cast(pa.string())
        table = pa.table({"text": text_array})
        all_tables.append(table)

    if not all_tables:
        raise ValueError("Could not read any .arrow files. They may be corrupted.")

    combined_table = pa.concat_tables(all_tables)    # all schemas are now identical
    ds = Dataset(combined_table)
    return ds


raw_dataset = load_arrow_data(DATASET_FOLDER)

print(f"\nDataset loaded successfully!")
print(f"Total samples : {len(raw_dataset):,}")
print(f"Columns       : {raw_dataset.column_names}")
print(f"\nFirst 3 samples preview:")
print("-" * 60)
for i in range(min(3, len(raw_dataset))):
    print(f"[{i}] {raw_dataset[i]['text'][:200]}")
    print()

Loading 4 raw .arrow file(s) with PyArrow...
  Reading: c4-train-00000-of-00093.arrow
    -> 79,568 rows | columns: ['text', 'timestamp', 'url']
  Reading: c4-train-00001-of-00093.arrow
    -> 81,261 rows | columns: ['text', 'timestamp', 'url']
  Reading: data-00000-of-00295.arrow
    -> 82,410 rows | columns: ['document_id', 'text']
  Reading: data-00001-of-00295.arrow
    -> 82,410 rows | columns: ['document_id', 'text']

Dataset loaded successfully!
Total samples : 325,649
Columns       : ['text']

First 3 samples preview:
------------------------------------------------------------
[0] দ্বিতীয় মেঘনা-গোমতী সেতু উদ্বোধন করলেন প্রধানমন্ত্রীDAILYSYLHET.COM | SYLHET NEWS | BANGLA NEWS
ডেইলি সিলেট ডট কম :: প্রকাশিত হয়েছে : মে ২৫, ২০১৯ | ১০:৫৫ পূর্বাহ্ন
নিউজ ডেস্ক:: প্রধানমন্ত্রী শেখ হাসিন

[1] অনুর্ধ্ব-২০ কোটিফ টুর্নামেন্টে আর্জেন্টিনা চ্যাম্পিয়ন | দৈনিক খুলনাঞ্চল
Home খেলাধুলা অনুর্ধ্ব-২০ কোটিফ টুর্নামেন্টে আর্জেন্টিনা চ্যাম্পিয়ন
অনুর্ধ্ব-২০ কোটিফ টুর্নামেন্টে আর্জেন্টিনা চ্যাম্পিয়ন
ফুট

In [8]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Bengali text cleaning function
# Removes: HTML, URLs, emails, non-Bengali characters,
#          extra whitespace, very short texts
# Keeps:   Bengali Unicode (U+0980–U+09FF), punctuation
# ─────────────────────────────────────────────────────────────

import re
import unicodedata

def clean_bengali_text(text):
    """Cleans a single Bengali text string. Returns None if unusable."""

    if not isinstance(text, str):
        return None
    text = text.strip()
    if len(text) == 0:
        return None

    text = re.sub(r'<[^>]+>', ' ', text)           # strip HTML tags
    text = re.sub(r'https?://\S+', ' ', text)      # strip URLs
    text = re.sub(r'www\.\S+',     ' ', text)
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)      # strip emails
    text = unicodedata.normalize('NFC', text)       # normalise Unicode

    # Keep Bengali block + basic punctuation + digits
    text = re.sub(
        r'[^\u0980-\u09FF'
        r'\u0020\u0009\u000A'
        r'\.,!?\;\:\"\(\)\-\u2013\u0964\u0965'
        r"'"
        r'0-9'
        r']+',
        ' ',
        text
    )

    text = re.sub(r'[ \t]+',   ' ',    text)
    text = re.sub(r'\n{3,}',   '\n\n', text)
    text = text.strip()

    if len(text) < 20:
        return None

    bengali_chars = len(re.findall(r'[\u0980-\u09FF]', text))
    total_chars   = len(text.replace(' ', ''))
    if total_chars > 0 and (bengali_chars / total_chars) < 0.5:
        return None

    return text


# Quick sanity test
test_sentences = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "<p>বাংলাদেশ</p> একটি সুন্দর দেশ। http://example.com",
    "This is English only, no Bengali here.",
    "   ",
    "বাংলা",
]

print("Cleaning test results:")
print("-" * 50)
for s in test_sentences:
    result = clean_bengali_text(s)
    print(f"Input  : {s[:70]}")
    print(f"Output : {result}")
    print()

Cleaning test results:
--------------------------------------------------
Input  : আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।
Output : আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।

Input  : <p>বাংলাদেশ</p> একটি সুন্দর দেশ। http://example.com
Output : বাংলাদেশ একটি সুন্দর দেশ।

Input  : This is English only, no Bengali here.
Output : None

Input  :    
Output : None

Input  : বাংলা
Output : None



In [9]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Apply cleaning to full dataset
# ─────────────────────────────────────────────────────────────

import datasets
datasets.disable_caching()

def clean_batch(examples):
    cleaned = [clean_bengali_text(t) for t in examples["text"]]
    return {"text": cleaned}

print("Cleaning dataset...")

cleaned_dataset = raw_dataset.map(
    clean_batch,
    batched=True,
    batch_size=500,
    desc="Cleaning text"
)

before_count = len(cleaned_dataset)
cleaned_dataset = cleaned_dataset.filter(
    lambda x: x["text"] is not None and len(x["text"]) >= 20,
    desc="Removing empty rows"
)
after_count = len(cleaned_dataset)

print(f"\nCleaning complete!")
print(f"Before : {before_count:,}")
print(f"After  : {after_count:,}")
print(f"Removed: {before_count - after_count:,}  ({(before_count - after_count) / before_count * 100:.1f}%)")

Cleaning dataset...


Removing empty rows: 100%|██████████| 325649/325649 [00:03<00:00, 102975.50 examples/s]


Cleaning complete!
Before : 325,649
After  : 324,110
Removed: 1,539  (0.5%)


In [10]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Remove duplicate texts
# Uses first 80 characters as a fingerprint
# ─────────────────────────────────────────────────────────────

print("Removing duplicates...")

seen_fingerprints = set()
unique_indices    = []

for i, example in enumerate(cleaned_dataset):
    fingerprint = example["text"][:80].strip()
    if fingerprint not in seen_fingerprints:
        seen_fingerprints.add(fingerprint)
        unique_indices.append(i)
    if (i + 1) % 100_000 == 0:
        print(f"  Checked {i+1:,} | Kept {len(unique_indices):,}")

dedup_dataset = cleaned_dataset.select(unique_indices)

print(f"\nDeduplication complete!")
print(f"Before : {len(cleaned_dataset):,}")
print(f"After  : {len(dedup_dataset):,}")
print(f"Removed: {len(cleaned_dataset) - len(dedup_dataset):,} duplicates")

Removing duplicates...
  Checked 100,000 | Kept 99,874
  Checked 200,000 | Kept 195,637
  Checked 300,000 | Kept 288,065

Deduplication complete!
Before : 324,110
After  : 310,717
Removed: 13,393 duplicates


In [11]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Save cleaned dataset to disk
# ─────────────────────────────────────────────────────────────

dedup_dataset.save_to_disk(CLEANED_OUTPUT)

sample_path = f"{CLEANED_OUTPUT}/sample_100.txt"
with open(sample_path, "w", encoding="utf-8") as f:
    for i in range(min(100, len(dedup_dataset))):
        f.write(dedup_dataset[i]["text"] + "\n\n" + "-"*60 + "\n\n")

print(f"Cleaned dataset saved  -> {CLEANED_OUTPUT}")
print(f"100-sample preview     -> {sample_path}")

Saving the dataset (4/4 shards): 100%|██████████| 310717/310717 [00:02<00:00, 146717.92 examples/s]

Cleaned dataset saved  -> ./BanglaLLM/cleaned
100-sample preview     -> ./BanglaLLM/cleaned/sample_100.txt


In [ ]:
# # ─────────────────────────────────────────────────────────────
# # CELL 8b — Preview cleaned samples
# # ─────────────────────────────────────────────────────────────

# import random

# NUM_SAMPLES = 20      # how many samples to show

# total = len(dedup_dataset)
# indices = random.sample(range(total), min(NUM_SAMPLES, total))

# print(f"Showing {len(indices)} random samples from {total:,} cleaned rows")
# print("=" * 70)

# for rank, idx in enumerate(indices, 1):
#     text = dedup_dataset[idx]["text"]
#     print(f"\n[{rank:02d}]  (index {idx:,} | {len(text)} chars)")
#     print("-" * 70)
#     print(text[:500])
#     if len(text) > 500:
#         print(f"... [{len(text) - 500} more chars]")

# print("\n" + "=" * 70)
# print(f"Total cleaned samples : {total:,}")

# # ── Length distribution ───────────────────────────────────────
# lengths = [len(dedup_dataset[i]["text"]) for i in range(min(5000, total))]

# print(f"\nText length stats (over {len(lengths):,} samples):")
# print(f"  Min    : {min(lengths):,} chars")
# print(f"  Max    : {max(lengths):,} chars")
# print(f"  Mean   : {sum(lengths) // len(lengths):,} chars")
# print(f"  Median : {sorted(lengths)[len(lengths)//2]:,} chars")

In [14]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — Train SentencePiece tokenizer on your Bengali corpus
# ─────────────────────────────────────────────────────────────
# pip install sentencepiece

import sentencepiece as spm
import os

SP_MODEL_PREFIX = f"{TOKENIZER_OUTPUT}/bangla_spm"   # saves .model + .vocab
VOCAB_SIZE      = 32000     # standard; increase to 52000 for richer coverage
SAMPLE_SIZE     = 500_000   # rows to train SP on (None = full dataset)

# ── Step 1: Write training text to a temp file ───────────────
SP_TRAIN_TXT = "./BanglaLLM/sp_train_input.txt"

print("Writing training text for SentencePiece...")
dedup_dataset = load_from_disk(CLEANED_OUTPUT)

if SAMPLE_SIZE and SAMPLE_SIZE < len(dedup_dataset):
    sp_data = dedup_dataset.select(range(SAMPLE_SIZE))
else:
    sp_data = dedup_dataset

with open(SP_TRAIN_TXT, "w", encoding="utf-8") as f:
    for example in sp_data:
        line = example["text"].replace("\n", " ").strip()
        if line:
            f.write(line + "\n")

print(f"Training file written: {SP_TRAIN_TXT}")
print(f"Lines written        : {len(sp_data):,}")

# ── Step 2: Train SentencePiece Unigram model ────────────────
print(f"\nTraining SentencePiece (vocab_size={VOCAB_SIZE:,})...")

spm.SentencePieceTrainer.train(
    input=SP_TRAIN_TXT,
    model_prefix=SP_MODEL_PREFIX,
    vocab_size=VOCAB_SIZE,
    model_type="unigram",           # best for Bengali conjuncts (যুক্তাক্ষর)
    character_coverage=0.9999,      # must be near 1.0 for non-Latin scripts
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>",
    byte_fallback=True,             # encodes unseen chars as UTF-8 bytes → no true OOV
    split_digits=True,              # tokenizes digits individually
    num_threads=os.cpu_count(),
    input_sentence_size=SAMPLE_SIZE or 1_000_000,
    shuffle_input_sentence=True,
)

print(f"\nSentencePiece model saved -> {SP_MODEL_PREFIX}.model")
print(f"Vocab file saved          -> {SP_MODEL_PREFIX}.vocab")

# ── Step 3: Quick test ────────────────────────────────────────
sp = spm.SentencePieceProcessor(model_file=f"{SP_MODEL_PREFIX}.model")

test_text = "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।"
tokens = sp.encode(test_text, out_type=str)
ids    = sp.encode(test_text)

print(f"\nTest sentence : {test_text}")
print(f"Tokens        : {tokens}")
print(f"Token IDs     : {ids}")
print(f"Vocab size    : {sp.get_piece_size():,}")

Writing training text for SentencePiece...
Training file written: ./BanglaLLM/sp_train_input.txt
Lines written        : 310,717

Training SentencePiece (vocab_size=32,000)...

SentencePiece model saved -> ./BanglaLLM/tokenizer/bangla_spm.model
Vocab file saved          -> ./BanglaLLM/tokenizer/bangla_spm.vocab

Test sentence : আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।
Tokens        : ['▁আমার', '▁সোনার', '▁বাংলা', ',', '▁আমি', '▁তোমায়', '▁ভালোবাসি', '।']
Token IDs     : [419, 2425, 604, 261, 403, 13069, 11107, 260]
Vocab size    : 32,000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./BanglaLLM/sp_train_input.txt
  input_format: 
  model_prefix: ./BanglaLLM/tokenizer/bangla_spm
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 0.9999
  input_sentence_size: 500000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 12
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surfac

In [22]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — Reload dataset + Tokenize using SentencePiece
# ─────────────────────────────────────────────────────────────

import gc
from datasets import load_from_disk
import sentencepiece as spm

MAX_LENGTH = 2048
TEST_SIZE  = 40_000          # set to None to use full dataset

# ── Reload SP model explicitly in this cell ──────────────────
SP_MODEL_PATH = f"{TOKENIZER_OUTPUT}/bangla_spm.model"
sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)

BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3

print(f"SP model loaded | vocab size: {sp.get_piece_size():,}")

# ── Redefine tokenize function here (never rely on kernel state) ──
def tokenize_function(examples):
    _sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)
    all_ids = []
    for text in examples["text"]:
        ids = [_sp.bos_id()] + _sp.encode(text) + [_sp.eos_id()]
        all_ids.append(ids)
    return {"input_ids": all_ids}

# ── Reload and optionally slice dataset ──────────────────────
print("\nReloading cleaned dataset...")
dedup_dataset = load_from_disk(CLEANED_OUTPUT)
print(f"Full dataset   : {len(dedup_dataset):,} samples")

if TEST_SIZE and TEST_SIZE < len(dedup_dataset):
    dedup_dataset = dedup_dataset.select(range(TEST_SIZE))
    print(f"Sliced to      : {len(dedup_dataset):,} samples (test run)")

print("\nTokenizing...")

cols_to_drop = [c for c in dedup_dataset.column_names if c != "text"]
dedup_dataset = dedup_dataset.remove_columns(cols_to_drop)

tokenized_dataset = dedup_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=500,
    remove_columns=["text"],
    num_proc=1,
    desc="Tokenizing",
)

del dedup_dataset
gc.collect()

print(f"\nTokenization complete!")
print(f"Samples  : {len(tokenized_dataset):,}")
print(f"Columns  : {tokenized_dataset.column_names}")

SP model loaded | vocab size: 32,000

Reloading cleaned dataset...
Full dataset   : 310,717 samples
Sliced to      : 40,000 samples (test run)

Tokenizing...


Tokenizing (num_proc=1): 100%|██████████| 40000/40000 [00:24<00:00, 1616.49 examples/s]


Tokenization complete!
Samples  : 40,000
Columns  : ['input_ids']


In [30]:
# ─────────────────────────────────────────────────────────────
# CELL 11 — Memory-safe streaming chunking
# Concatenates all token sequences, then slices into
# fixed MAX_LENGTH windows. Writes directly to .arrow file
# to avoid holding everything in RAM.
# ─────────────────────────────────────────────────────────────
import os
if os.path.exists(CHUNK_FILE):
    os.remove(CHUNK_FILE)
    print(f"Deleted corrupt file: {CHUNK_FILE}")

import pyarrow.ipc as ipc

BATCH_SIZE = 2000
CHUNK_FILE = f"{FINAL_OUTPUT}/chunks.arrow"

schema = pa.schema([
    ("input_ids", pa.list_(pa.int32())),
    ("labels",    pa.list_(pa.int32())),
])

total_chunks_written = 0
leftover = []

print(f"Streaming chunking -> {CHUNK_FILE}")
print(f"Input samples : {len(tokenized_dataset):,}")

with ipc.new_file(CHUNK_FILE, schema) as writer:
    for start in range(0, len(tokenized_dataset), BATCH_SIZE):
        end   = min(start + BATCH_SIZE, len(tokenized_dataset))
        batch = tokenized_dataset[start:end]

        for seq in batch["input_ids"]:
            leftover.extend(seq)

        n_chunks = len(leftover) // MAX_LENGTH

        if n_chunks > 0:
            usable   = leftover[: n_chunks * MAX_LENGTH]
            leftover = leftover[n_chunks * MAX_LENGTH :]
            rows     = [usable[i : i + MAX_LENGTH]
                        for i in range(0, len(usable), MAX_LENGTH)]

            writer.write_batch(pa.record_batch({
                "input_ids": pa.array(rows, type=pa.list_(pa.int32())),
                "labels":    pa.array(rows, type=pa.list_(pa.int32())),
            }, schema=schema))

            total_chunks_written += n_chunks
            del rows, usable

        del batch
        gc.collect()

        print(f"  Processed {end:,}/{len(tokenized_dataset):,} "
              f"| chunks: {total_chunks_written:,}", end="\r")

print(f"\n\nDone!")
print(f"Chunks written         : {total_chunks_written:,}")
print(f"Total tokens           : {total_chunks_written * MAX_LENGTH:,}")
print(f"Leftover tokens skipped: {len(leftover):,}")

# ── Load back using ipc.open_file (matches how it was written) ──
import pyarrow as pa
import pyarrow.ipc as ipc
from datasets import Dataset

print("Loading chunks...")

source        = pa.memory_map(CHUNK_FILE, "r")
reader        = ipc.open_file(source)
table         = reader.read_all()
chunked_dataset = Dataset(table)

print(f"Dataset ready : {len(chunked_dataset):,} chunks")
print(f"Sample check  : input_ids[0] length = {len(chunked_dataset[0]['input_ids'])}")


Deleted corrupt file: ./BanglaLLM/final_dataset/chunks.arrow
Streaming chunking -> ./BanglaLLM/final_dataset/chunks.arrow
Input samples : 40,000
  Processed 40,000/40,000 | chunks: 9,505

Done!
Chunks written         : 9,505
Total tokens           : 19,466,240
Leftover tokens skipped: 958
Loading chunks...
Dataset ready : 9,505 chunks
Sample check  : input_ids[0] length = 2048


In [31]:
# ─────────────────────────────────────────────────────────────
# CELL 12 — Define scaled-down LLaMA 3 config for RTX 3050
# Full 8B needs ~80GB VRAM. This fits in 4-8GB.
# ─────────────────────────────────────────────────────────────

from transformers import LlamaConfig, LlamaForCausalLM
import sentencepiece as spm
import torch

SP_MODEL_PATH = f"{TOKENIZER_OUTPUT}/bangla_spm.model"
sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)
VOCAB_SIZE = sp.get_piece_size()

# ── Model size options — pick ONE ────────────────────────────
# Uncomment the config that matches your available VRAM

# ~80M params — safe for 4GB VRAM, fast iteration
config = LlamaConfig(
    vocab_size           = VOCAB_SIZE,
    hidden_size          = 512,
    intermediate_size    = 1024,
    num_hidden_layers    = 8,
    num_attention_heads  = 8,
    num_key_value_heads  = 4,        # Grouped Query Attention (GQA) like LLaMA 3
    max_position_embeddings = 2048,
    rms_norm_eps         = 1e-5,
    rope_theta           = 500000.0, # LLaMA 3 default
    attention_bias       = False,
    mlp_bias             = False,
    hidden_act           = "silu",   # SwiGLU
    pad_token_id         = 0,
    bos_token_id         = 2,
    eos_token_id         = 3,
    tie_word_embeddings  = False,
)

# ~300M params — for 8GB VRAM (comment out above and use this)
# config = LlamaConfig(
#     vocab_size           = VOCAB_SIZE,
#     hidden_size          = 1024,
#     intermediate_size    = 2048,
#     num_hidden_layers    = 16,
#     num_attention_heads  = 16,
#     num_key_value_heads  = 8,
#     max_position_embeddings = 2048,
#     rms_norm_eps         = 1e-5,
#     rope_theta           = 500000.0,
#     attention_bias       = False,
#     mlp_bias             = False,
#     hidden_act           = "silu",
#     pad_token_id         = 0,
#     bos_token_id         = 2,
#     eos_token_id         = 3,
#     tie_word_embeddings  = False,
# )

# ── Initialize model with RANDOM weights (no pretrained) ─────
model = LlamaForCausalLM(config)
model_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model initialized from scratch (random weights)")
print(f"Parameters : {model_params:.1f}M")
print(f"Vocab size : {VOCAB_SIZE:,}")
print(f"Architecture: LLaMA 3 (RMSNorm + RoPE + SwiGLU + GQA)")

Model initialized from scratch (random weights)
Parameters : 51.7M
Vocab size : 32,000
Architecture: LLaMA 3 (RMSNorm + RoPE + SwiGLU + GQA)


In [33]:
# ─────────────────────────────────────────────────────────────
# CELL 13 — Load chunked dataset + build DataCollator
# ─────────────────────────────────────────────────────────────

import pyarrow as pa
import pyarrow.ipc as ipc
from datasets import Dataset
import torch
from torch.utils.data import Dataset as TorchDataset

# ── Load using ipc.open_file (matches how Cell 11 wrote it) ──
print("Loading chunked dataset...")
source          = pa.memory_map(CHUNK_FILE, "r")
table           = ipc.open_file(source).read_all()
chunked_dataset = Dataset(table)

print(f"Total chunks : {len(chunked_dataset):,}")
print(f"Tokens total : {len(chunked_dataset) * MAX_LENGTH:,}")

# ── Train / eval split ────────────────────────────────────────
split         = chunked_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"\nTrain chunks : {len(train_dataset):,}")
print(f"Eval  chunks : {len(eval_dataset):,}")

# ── Torch-compatible wrapper ──────────────────────────────────
class BengaliCLMDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = torch.tensor(self.data[idx]["input_ids"], dtype=torch.long)
        return {"input_ids": ids, "labels": ids.clone()}

train_torch = BengaliCLMDataset(train_dataset)
eval_torch  = BengaliCLMDataset(eval_dataset)

print(f"\nTorch datasets ready.")
print(f"Sample shape : {train_torch[0]['input_ids'].shape}")

Loading chunked dataset...
Total chunks : 9,505
Tokens total : 19,466,240

Train chunks : 9,029
Eval  chunks : 476

Torch datasets ready.
Sample shape : torch.Size([2048])


In [35]:
# ─────────────────────────────────────────────────────────────
# CELL 14 — Training arguments with failsafe checkpointing
# ─────────────────────────────────────────────────────────────

from transformers import TrainingArguments
import math
import torch

EPOCHS         = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 16
LR             = 3e-4
WARMUP_RATIO   = 0.05
OUTPUT_DIR     = "./BanglaLLM/checkpoints"

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,

    # ── Batch size & gradient accumulation ───────────────────
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,

    # ── Precision ─────────────────────────────────────────────
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),

    # ── Optimizer & scheduler ─────────────────────────────────
    learning_rate               = LR,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = 0.1,
    max_grad_norm               = 1.0,

    # ── Epochs & saving ───────────────────────────────────────
    num_train_epochs            = EPOCHS,
    save_strategy               = "steps",
    save_steps                  = 100,        # ← save every 100 steps (more frequent = safer)
    save_total_limit            = 5,          # ← keep last 5 checkpoints
    save_on_each_node           = True,

    # ── Evaluation ────────────────────────────────────────────
    eval_strategy               = "steps",
    eval_steps                  = 100,

    # ── FAILSAFE: always resume from latest checkpoint ────────
    resume_from_checkpoint      = True,       # ← auto-resumes if checkpoint exists

    # ── Logging ───────────────────────────────────────────────
    logging_dir                 = "./BanglaLLM/logs",
    logging_steps               = 10,
    logging_first_step          = True,
    report_to                   = "none",

    # ── Memory optimizations ──────────────────────────────────
    gradient_checkpointing      = True,
    optim                       = "adamw_torch_fused",
    dataloader_pin_memory       = True,
    dataloader_num_workers      = 2,

    seed                        = 42,
)

steps_per_epoch = math.ceil(len(train_torch) / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS

print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps per epoch      : {steps_per_epoch:,}")
print(f"Total steps          : {total_steps:,}")
print(f"Saves every          : 100 steps")
print(f"Checkpoints kept     : 5")
print(f"Precision            : {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"Device               : {torch.cuda.get_device_name(0)}")
print(f"VRAM available       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Effective batch size : 32
Steps per epoch      : 283
Total steps          : 849
Saves every          : 100 steps
Checkpoints kept     : 5
Precision            : bf16
Device               : NVIDIA GeForce RTX 3050
VRAM available       : 8.2 GB


In [36]:
# ─────────────────────────────────────────────────────────────
# CELL 15 — Train with automatic resume
# Just re-run this cell after any interruption.
# It will find the latest checkpoint and continue from there.
# ─────────────────────────────────────────────────────────────

import os
import glob
from transformers import Trainer

def get_latest_checkpoint(output_dir):
    """Returns path to latest checkpoint, or None if no checkpoints exist."""
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not checkpoints:
        return None
    # Sort by step number
    latest = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    return latest

# ── Detect existing checkpoint ───────────────────────────────
latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint:
    print(f"Checkpoint found  : {latest_checkpoint}")
    print(f"Training will RESUME from step {latest_checkpoint.split('-')[-1]}")
else:
    print("No checkpoint found — starting fresh training")

print("-" * 60)

# ── Build trainer ─────────────────────────────────────────────
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_torch,
    eval_dataset  = eval_torch,
)

# ── Train (pass checkpoint path if found) ────────────────────
try:
    trainer.train(resume_from_checkpoint=latest_checkpoint)
    print("\nTraining completed successfully!")

except KeyboardInterrupt:
    print("\nInterrupted by user — saving emergency checkpoint...")
    trainer.save_model(os.path.join(OUTPUT_DIR, "emergency-checkpoint"))
    print(f"Emergency checkpoint saved -> {OUTPUT_DIR}/emergency-checkpoint")
    print("Re-run this cell to resume.")

except Exception as e:
    print(f"\nUnexpected error: {e}")
    print("Saving emergency checkpoint...")
    try:
        trainer.save_model(os.path.join(OUTPUT_DIR, "emergency-checkpoint"))
        print(f"Emergency checkpoint saved -> {OUTPUT_DIR}/emergency-checkpoint")
    except Exception as save_err:
        print(f"Could not save checkpoint: {save_err}")
    raise

No checkpoint found — starting fresh training
------------------------------------------------------------


Step,Training Loss,Validation Loss
100,7.959619,7.910243
200,7.040546,7.023021
300,6.547686,6.548532
400,6.274746,6.245734
500,6.017742,6.047724
600,5.866327,5.926543
700,5.779115,5.866000
800,5.772935,5.845604
849,5.785955,5.844647


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.78it/s]



Training completed successfully!


In [37]:
# ─────────────────────────────────────────────────────────────
# CELL 16 — Save final model + config
# ─────────────────────────────────────────────────────────────

FINAL_MODEL_DIR = "./BanglaLLM/final_model"

trainer.save_model(FINAL_MODEL_DIR)
config.save_pretrained(FINAL_MODEL_DIR)

print(f"Model saved -> {FINAL_MODEL_DIR}")

# ── Quick generation test ─────────────────────────────────────
from transformers import pipeline
import sentencepiece as spm

sp    = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)
model.eval()

prompt      = "বাংলাদেশ একটি"
input_ids   = torch.tensor([[sp.bos_id()] + sp.encode(prompt)]).to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens  = 50,
        do_sample       = True,
        temperature     = 0.8,
        top_p           = 0.9,
        eos_token_id    = sp.eos_id(),
        pad_token_id    = 0,
    )

generated = sp.decode(output[0].tolist())
print(f"\nPrompt    : {prompt}")
print(f"Generated : {generated}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.43it/s]


Model saved -> ./BanglaLLM/final_model

Prompt    : বাংলাদেশ একটি
Generated : বাংলাদেশ একটি কথা। তার কথা না। তিনি বলেন, আমাদের এক নারীর জন্য একটি প্রশ্ন করছি। তিনি বলেন, তার সাথে আমি তাঁর এক ভাই। তার বয়স হলে আমি তো আমার কাছে বলেছি। তিনি বলেন, আমি একজন বলেন, আমি তোমাদের মধ্যে কথা না। আমি
